In [ ]:
# @title Setup Env
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers chromadb \
                langchain langchain-huggingface langchain-chroma langchain-community langchain-ollama \
                --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/

# 2-stage IR with `langchain_classic` compressor/reranker

After many days combing through the LangChain docs, outdated tutorials, and mutually contradicting LLMs, I figured why I was having so much trouble adding a second stage my base information retrieval pipeline.

In `langchain > 1.x` a lot has changed from the older versions, including the setup of a HF cross-encoder reranker. Some of the classes used in "the old way" now are part of `langchain_classic`, which I guess means they are deprecated until further notice?

Anyway, I still wanted to give it a shot at assembling the pipeline myself from the sources I found (and extensive help from LLMs), even though it's probably (definitely) no longer the recommended way to go with newer LangChain versions.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors.cross_encoder_rerank import CrossEncoderReranker
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer

import torch
import hashlib
import os

In [ ]:
# @title Load pretrained embedder model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}.")
model_kwargs = {'device': device, # Use 'cuda:0' to specify a GPU, if multiple available
                # 'device_map': 'auto' # Use this to automatically spread the model across multiple GPUs, or
                #       to offload parts of the model from the GPU to the CPU if it's too big for VRAM. DON'T SET BOTH
                }

model_name = "PORTULAN/serafim-100m-portuguese-pt-sentence-encoder"
embedding_model = HuggingFaceEmbeddings(model_name=model_name,
                                        model_kwargs=model_kwargs)
# Change max length for longer chunks (BERT-style models support a max sequence len of 512)
embedding_model._client.max_seq_length = 512

# Load tokenizer separately to use `RecursiveCharacterTextSplitter.from_huggingface_tokenizer`
embedding_tokenizer = AutoTokenizer.from_pretrained(model_name)

embedding_model

Using device cpu.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

HuggingFaceEmbeddings(model_name='PORTULAN/serafim-100m-portuguese-pt-sentence-encoder', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
# @title Instantiate and populate vector DB with chunked dummy documents


print("Starting Chroma...")
# Setup ChromaDB
vector_store = Chroma(
    collection_name="my_collection",
    embedding_function=embedding_model, # Embeddings are calculated on the fly
    persist_directory="./chroma_langchain_db", # Save data locally, or use host=[...] to connect to an existing ChromaDB server
    # host="localhost",
)

# Download "database"
!curl -o glossary.md https://raw.githubusercontent.com/GabrielvanderSchmidt/SENAC-DLGlossary/refs/heads/main/glossary.md

print("Doc-in' n' chunkin'...")
documents = [file for file in os.listdir(".") if os.path.isfile(file)]
documents_text = []
documents_meta = []
i = 0
for doc in documents:
    with open(doc, "r") as file:
        documents_text.append(file.read())
        documents_meta.append({"document_id" : str(i),
                               "tags" : ["learning", "AI"],
                               "source" : doc})
        i = i+1

# Convert strings to Document objects
documents = [Document(page_content=text, metadata=meta) for text, meta in zip(documents_text, documents_meta)]

# Chunk docs
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embedding_tokenizer, # Load tokenizer to calculate chunk length
    chunk_size=200, # Chunk size (tokens) -- must be smaller than embedder sequence length
    chunk_overlap=30, # Chunk overlap (tokens) so context isn't lost at the "edges" of chunks
    add_start_index=True, # (Try to) Track index in original document
)

all_splits = text_splitter.split_documents(documents)
print(f"Documents split in {len(all_splits)} chunks.")

# Ids can be generated beforehand, or created by the `add_documents` call
# Let's generate them with our hashing function
def generate_ids(chunks):
    text_chunk = [chunk.page_content for chunk in chunks]
    encoded_chunks = [chunk.encode("utf-8") for chunk in text_chunk]
    hashes = [hashlib.md5(chunk).hexdigest() for chunk in encoded_chunks]
    return hashes

print("Embedding and pushing to vector DB...")
ids = generate_ids(all_splits)
vector_store.add_documents(documents=all_splits, ids=ids) # Outputs added ids

# Check chunking output
vector_store.get(ids=ids[0])

Starting Chroma...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13632  100 13632    0     0  46025      0 --:--:-- --:--:-- --:--:-- 46054
Doc-in' n' chunkin'...
Documents split in 33 chunks.
Embedding and pushing to vector DB...


{'ids': ['cc1628ceebfed1c1767838a68ad9f81e'],
 'embeddings': None,
 'documents': ['# Estrutura da Rede'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'tags': ['learning', 'AI'],
   'start_index': 0,
   'document_id': '0',
   'source': 'glossary.md'}]}

In [ ]:
# @title Load pretrained cross-encoder model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}.")
model_kwargs = {'device': device, # Use 'cuda:0' to specify a GPU, if multiple available
                # 'device_map': 'auto' # Use this to automatically spread the model across multiple GPUs, or to offload
                #        parts of the model from the GPU to the CPU if it's too big for VRAM. DON'T SET BOTH
                }

# model_name = "PORTULAN/albertina-1b5-portuguese-ptbr-encoder" # Final model we will want to use
model_name =  "cross-encoder/ms-marco-MiniLM-L-6-v2" # Placeholder model for now
cross_encoder_model = HuggingFaceCrossEncoder(model_name=model_name,
                                              model_kwargs=model_kwargs)
cross_encoder_model

Using device cpu.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceCrossEncoder(client=CrossEncoder(
  (model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-5): 6 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=384, out_features=384, bias=True)
                (key): Linear(in_features=384, out_features=384, bias=True)
                (value): Linear(in_features=384, out_features=384, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_feat

In [ ]:
# @title Setup two-stage retrieval pipeline


# Set vector DB + embedding model as a retriever (1st stage)
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20} # Retrieve a massive candidate pool for subsequent reranking
) # In this stage, we maximize recall (minimize false negatives - important context left out)

# Initialize reranking object (2nd stage)
reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=3 # Retain only the 3 most strictly relevant chunks
) # In this stage, we maximize precision (minimize false positives - trash let in)

# Construct the unified two-stage retrieval pipeline
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

for i, doc in enumerate(compression_retriever.invoke("Qual é o componente mais fundamental de redes neurais?")):
    print(f"Top {i+1} match:")
    print(doc.page_content)

Top 1 match:
É o tipo mais simples de rede neural **feedforward**, e hoje é usado como sinônimo para **neurônio artificial**, que é o bloco fundamental de **redes neurais artificiais**. Neurônios artificiais são agrupados lateralmente em **camadas densas**, e estas são empilhadas verticalmente para formar **redes neurais multicamadas**. Consiste de uma soma das entradas ponderadas pelos **pesos/parâmetros** conectados $$u = w_0 \cdot X_0 + … + w_n \cdot X_n + b$$ seguida de uma **função de ativação** não-linear $$g(u)$$. Um neurônio artificial é equivalente a uma regressão logística quando usando **função de ativação sigmóide**.
Top 2 match:
#### Rede Convolucional (Convolutional Neural Network - CNN)
Nome genérico para **redes neurais** que possuem **camadas convolucionais** como seu mecanismo interno principal. Corresponde à maioria das arquiteturas voltadas para o **tratamento de imagens**, mas também aparecendo no **processamento de áudio** (convoluções 1D sobre o sinal de áudio br